In [242]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrices
from statsmodels.sandbox.tsa.fftarma import ArmaFft

In [243]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter('ignore', InterpolationWarning)

In [244]:
df = pd.read_csv('modified_data/wdi.csv', index_col = 0)
df = df.loc[
    df['Series'] == "Total greenhouse gas emissions excluding LULUCF per capita (t CO2e/capita)", 
    ['Country', 'year', 'value']
]


In [245]:
df

,Country,year,value
32550,Zimbabwe,1990,3.397775
32551,Zambia,1990,2.037472
32555,Viet Nam,1990,1.402883
32556,"Venezuela, RB",1990,8.257977
32558,Uzbekistan,1990,7.973694
...,...,...,...
66393,Armenia,2021,3.682274
66394,Argentina,2021,8.194648
66396,Angola,2021,1.928677
66399,Algeria,2021,5.493078


In [246]:
all_TS = df.pivot(index='year', columns='Country', values='value')

In [247]:
all_TS

Country,Albania,Algeria,Angola,Argentina,Armenia,Australia,Austria,Azerbaijan,Bahrain,Bangladesh,...,Ukraine,United Arab Emirates,United Kingdom,United States,Uruguay,Uzbekistan,"Venezuela, RB",Viet Nam,Zambia,Zimbabwe
year,,,,,,,,,,,,,,,,,,,,,
1990,3.498632,5.201907,2.931657,7.725790,6.842884,26.996569,10.442702,9.431570,56.462062,0.670716,...,18.787107,41.883291,13.592383,24.902860,9.309809,7.973694,8.257977,1.402883,2.037472,3.397775
1991,2.771099,5.110738,2.777812,7.731196,6.881477,26.817099,10.886755,9.057981,56.884775,0.659004,...,17.611385,43.798496,13.728869,24.355254,9.832133,7.752135,8.316408,1.435843,1.856220,3.300097
1992,1.900624,5.366717,2.628957,7.684405,3.925097,26.633865,10.070339,7.495100,63.562574,0.669665,...,15.645357,41.339036,13.384681,24.398846,10.365920,7.107589,8.173746,1.472587,1.881081,3.294421
1993,1.924000,5.349047,2.855704,7.635627,2.223534,26.655226,9.987448,6.539908,65.879579,0.672253,...,13.850475,42.326266,13.001575,24.547482,10.544158,6.975546,8.200585,1.547663,1.782259,2.833894
1994,2.153460,5.176934,3.577852,7.693512,1.569543,26.629241,10.004388,5.726060,64.069578,0.684206,...,11.780926,42.496869,12.743316,24.668365,10.540135,6.694384,8.665803,1.626551,1.677046,2.747604
1995,2.054531,5.357291,3.987313,7.673785,1.802042,26.784063,10.351752,5.552626,62.157462,0.721533,...,11.562235,42.710035,12.663536,24.631014,10.595277,6.204671,8.609207,1.718190,1.595786,2.711142
1996,1.983060,5.219842,4.625123,7.782154,1.525344,27.178087,10.840371,4.800850,62.840890,0.720569,...,10.113517,41.994692,12.995455,24.974857,10.990998,6.278859,9.142329,1.808445,1.517039,2.801361
1997,1.756260,4.801986,4.589366,7.824840,1.756714,27.580029,10.729187,4.663694,56.782801,0.727343,...,9.767864,39.351497,12.611724,25.711616,10.849599,6.106623,9.170638,1.877650,1.436874,2.655353
1998,1.850358,4.696868,4.822166,7.840206,1.784877,28.437700,10.733467,4.673904,57.819067,0.722224,...,9.324615,36.816349,12.487046,25.575560,10.653849,6.669408,9.207889,1.987676,1.566079,2.682362


In [248]:
train_TS = all_TS.loc[all_TS.index < 2012]

In [249]:
test_TS = all_TS.loc[all_TS.index >= 2012]

In [250]:
all_TS.to_csv('modified_data/ghg_TimeSeries.csv')
train_TS.to_csv('modified_data/ghg_TimeSeries_train.csv')
test_TS.to_csv('modified_data/ghg_TimeSeries_test.csv')

ADF i KPSS

In [251]:
from statsmodels.tsa.stattools import adfuller


def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC", result_object=True)
    dfoutput = pd.Series(
        [dftest.statistic, dftest.pvalue, dftest.lags, int(dftest.nobs)],
        index=[
            "Test Statistic",
            "pvalue",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest.critical_values.items():
        dfoutput[f"Critical Value ({key})"] = value
    return(dfoutput.pvalue)

In [252]:
from statsmodels.tsa.stattools import kpss


def kpss_test(timeseries):
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto", result_object=True)
    kpss_output = pd.Series(
        [kpsstest.statistic, kpsstest.pvalue, kpsstest.lags], index=["Test Statistic", "pvalue", "Lags Used"]
    )
    for key, value in kpsstest.critical_values.items():
        kpss_output[f"Critical Value ({key})"] = value
    return(kpss_output.pvalue)

In [253]:
train_TS

Country,Albania,Algeria,Angola,Argentina,Armenia,Australia,Austria,Azerbaijan,Bahrain,Bangladesh,...,Ukraine,United Arab Emirates,United Kingdom,United States,Uruguay,Uzbekistan,"Venezuela, RB",Viet Nam,Zambia,Zimbabwe
year,,,,,,,,,,,,,,,,,,,,,
1990,3.498632,5.201907,2.931657,7.725790,6.842884,26.996569,10.442702,9.431570,56.462062,0.670716,...,18.787107,41.883291,13.592383,24.902860,9.309809,7.973694,8.257977,1.402883,2.037472,3.397775
1991,2.771099,5.110738,2.777812,7.731196,6.881477,26.817099,10.886755,9.057981,56.884775,0.659004,...,17.611385,43.798496,13.728869,24.355254,9.832133,7.752135,8.316408,1.435843,1.856220,3.300097
1992,1.900624,5.366717,2.628957,7.684405,3.925097,26.633865,10.070339,7.495100,63.562574,0.669665,...,15.645357,41.339036,13.384681,24.398846,10.365920,7.107589,8.173746,1.472587,1.881081,3.294421
1993,1.924000,5.349047,2.855704,7.635627,2.223534,26.655226,9.987448,6.539908,65.879579,0.672253,...,13.850475,42.326266,13.001575,24.547482,10.544158,6.975546,8.200585,1.547663,1.782259,2.833894
1994,2.153460,5.176934,3.577852,7.693512,1.569543,26.629241,10.004388,5.726060,64.069578,0.684206,...,11.780926,42.496869,12.743316,24.668365,10.540135,6.694384,8.665803,1.626551,1.677046,2.747604
1995,2.054531,5.357291,3.987313,7.673785,1.802042,26.784063,10.351752,5.552626,62.157462,0.721533,...,11.562235,42.710035,12.663536,24.631014,10.595277,6.204671,8.609207,1.718190,1.595786,2.711142
1996,1.983060,5.219842,4.625123,7.782154,1.525344,27.178087,10.840371,4.800850,62.840890,0.720569,...,10.113517,41.994692,12.995455,24.974857,10.990998,6.278859,9.142329,1.808445,1.517039,2.801361
1997,1.756260,4.801986,4.589366,7.824840,1.756714,27.580029,10.729187,4.663694,56.782801,0.727343,...,9.767864,39.351497,12.611724,25.711616,10.849599,6.106623,9.170638,1.877650,1.436874,2.655353
1998,1.850358,4.696868,4.822166,7.840206,1.784877,28.437700,10.733467,4.673904,57.819067,0.722224,...,9.324615,36.816349,12.487046,25.575560,10.653849,6.669408,9.207889,1.987676,1.566079,2.682362


* ze wzgledu na ograniczenia testu kpss - wyniki p-value mimo wyzszej wartosci sa ucinane na 0,1 (wylaczone ostrzezeania)

In [254]:
stacjonarnosc = pd.DataFrame(columns=['Kraj', 'ADF p-value', 'KPSS p-value'])

for kraj in train_TS.columns:
    print("Wyniki dla: " + kraj)
    print(adf_test(train_TS[kraj]))
    print(kpss_test(train_TS[kraj]))
    stacjonarnosc.loc[len(stacjonarnosc)] = [kraj, adf_test(train_TS[kraj]), kpss_test(train_TS[kraj])]



Wyniki dla: Albania
Results of Dickey-Fuller Test:
0.9847289259257909
Results of KPSS Test:
0.1
Results of Dickey-Fuller Test:
Results of KPSS Test:
Wyniki dla: Algeria
Results of Dickey-Fuller Test:
0.9654034264007433
Results of KPSS Test:
0.08290503439519888
Results of Dickey-Fuller Test:
Results of KPSS Test:
Wyniki dla: Angola
Results of Dickey-Fuller Test:
0.813274262085559
Results of KPSS Test:
0.1
Results of Dickey-Fuller Test:
Results of KPSS Test:
Wyniki dla: Argentina
Results of Dickey-Fuller Test:
0.9951389057645206
Results of KPSS Test:
0.03433487366932779
Results of Dickey-Fuller Test:
Results of KPSS Test:
Wyniki dla: Armenia
Results of Dickey-Fuller Test:
0.9907734310417606
Results of KPSS Test:
0.1
Results of Dickey-Fuller Test:
Results of KPSS Test:
Wyniki dla: Australia
Results of Dickey-Fuller Test:
3.49896925092162e-07
Results of KPSS Test:
0.0927949382949576
Results of Dickey-Fuller Test:
Results of KPSS Test:
Wyniki dla: Austria
Results of Dickey-Fuller Test:
0.90

In [257]:
stacjonarnosc

,Kraj,ADF p-value,KPSS p-value
0,Albania,0.984729,0.100000
1,Algeria,0.965403,0.082905
2,Angola,0.813274,0.100000
3,Argentina,0.995139,0.034335
4,Armenia,0.990773,0.100000
...,...,...,...
133,Uzbekistan,0.041120,0.100000
134,"Venezuela, RB",0.910430,0.048039
135,Viet Nam,0.998684,0.010000
136,Zambia,0.026153,0.013227
